<h1>SQLAlchemy Tutorial<h1/>

In [1]:
import sqlalchemy

In [2]:
sqlalchemy.__version__

'2.0.46'

In [3]:
from sqlalchemy import create_engine, text

In [4]:
engine = create_engine("sqlite+pysqlite:///:memory:", echo=True)

In [5]:
with engine.connect() as conn:
    result =  conn.execute(text("select 'hello world'"))
    print(result.all())

2026-01-25 12:35:08,351 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:08,353 INFO sqlalchemy.engine.Engine select 'hello world'
2026-01-25 12:35:08,353 INFO sqlalchemy.engine.Engine [generated in 0.00198s] ()
[('hello world',)]
2026-01-25 12:35:08,354 INFO sqlalchemy.engine.Engine ROLLBACK


In [6]:
# Commit as you go"
with engine.connect() as conn:
    conn.execute(text("CREATE TABLE some_table (x int, y int)"))
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"), 
        [{"x": 1, "y": 1}, {"x": 2, "y": 4}],
    )
    conn.commit()

2026-01-25 12:35:08,380 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:08,381 INFO sqlalchemy.engine.Engine CREATE TABLE some_table (x int, y int)
2026-01-25 12:35:08,382 INFO sqlalchemy.engine.Engine [generated in 0.00197s] ()
2026-01-25 12:35:08,385 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-25 12:35:08,387 INFO sqlalchemy.engine.Engine [generated in 0.00168s] [(1, 1), (2, 4)]
2026-01-25 12:35:08,388 INFO sqlalchemy.engine.Engine COMMIT


In [7]:
# begins once#
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 6, "y": 8}, {"x": 9, "y": 10}],
    )

2026-01-25 12:35:08,416 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:08,418 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-25 12:35:08,419 INFO sqlalchemy.engine.Engine [cached since 0.0331s ago] [(6, 8), (9, 10)]
2026-01-25 12:35:08,475 INFO sqlalchemy.engine.Engine COMMIT


In [8]:
# begins once#
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (9, 10)")
            )

2026-01-25 12:35:08,492 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:08,494 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (9, 10)
2026-01-25 12:35:08,494 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ()
2026-01-25 12:35:08,496 INFO sqlalchemy.engine.Engine COMMIT


In [9]:
# select statement#
with engine.begin() as conn:
    query_result = conn.execute(text("SELECT * FROM some_table"))
    print(query_result.all())

2026-01-25 12:35:08,524 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:08,526 INFO sqlalchemy.engine.Engine SELECT * FROM some_table
2026-01-25 12:35:08,527 INFO sqlalchemy.engine.Engine [generated in 0.00110s] ()
[(1, 1), (2, 4), (6, 8), (9, 10), (9, 10)]
2026-01-25 12:35:08,530 INFO sqlalchemy.engine.Engine COMMIT


In [10]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table"))
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-25 12:35:08,554 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:08,555 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table
2026-01-25 12:35:08,556 INFO sqlalchemy.engine.Engine [generated in 0.00222s] ()
x: 1 y: 1
x: 2 y: 4
x: 6 y: 8
x: 9 y: 10
x: 9 y: 10
2026-01-25 12:35:08,558 INFO sqlalchemy.engine.Engine ROLLBACK


<h2>Sending Parameters<h2/>

In [11]:
# return value of y where its value is greater than a specific value)

with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table WHERE y > :y"), {"y": 8})
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-25 12:35:08,588 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:08,589 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ?
2026-01-25 12:35:08,590 INFO sqlalchemy.engine.Engine [generated in 0.00247s] (8,)
x: 9 y: 10
x: 9 y: 10
2026-01-25 12:35:08,592 INFO sqlalchemy.engine.Engine ROLLBACK


<h2>Sending Multiple Parameters<h2/>

In [12]:
# inserting multiple records in a sql statement

with engine.connect() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"), 
        [{"x": 11, "y": 12}, {"x": 13, "y": 14}]
    )
    conn.commit()

2026-01-25 12:35:08,619 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:08,620 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-25 12:35:08,621 INFO sqlalchemy.engine.Engine [cached since 0.2352s ago] [(11, 12), (13, 14)]
2026-01-25 12:35:08,622 INFO sqlalchemy.engine.Engine COMMIT


<h2>Executing with an ORM Session<h2/>

In [13]:
from sqlalchemy.orm import Session

In [14]:
stmt = text("SELECT x, y FROM some_table WHERE y > :y ORDER BY x, y")
with Session(engine) as session:
    result = session.execute(stmt, {"y": 6})
    for row in result:
        print(f"x: {row.x}, y: {row.y}")

2026-01-25 12:35:08,816 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:08,818 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ? ORDER BY x, y
2026-01-25 12:35:08,819 INFO sqlalchemy.engine.Engine [generated in 0.00135s] (6,)
x: 6, y: 8
x: 9, y: 10
x: 9, y: 10
x: 11, y: 12
x: 13, y: 14
2026-01-25 12:35:08,822 INFO sqlalchemy.engine.Engine ROLLBACK


In [15]:
# commit #

with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11},{"x": 13, "y": 15}],
    )
    session.commit()

2026-01-25 12:35:08,847 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:08,848 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-25 12:35:08,849 INFO sqlalchemy.engine.Engine [generated in 0.00109s] [(11, 9), (15, 13)]
2026-01-25 12:35:08,852 INFO sqlalchemy.engine.Engine COMMIT


In [16]:
with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11}, {"x": 13, "y": 15}]
    )
    session.commit()

2026-01-25 12:35:08,879 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:08,880 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-25 12:35:08,882 INFO sqlalchemy.engine.Engine [cached since 0.03315s ago] [(11, 9), (15, 13)]
2026-01-25 12:35:08,883 INFO sqlalchemy.engine.Engine COMMIT


<h2>Setting up MetaData with Table objects<h2/>

In [17]:
from sqlalchemy import MetaData
metadata_obj = MetaData()

In [18]:
from sqlalchemy import Table, Column, Integer, String
user_table = Table(
    "user_account",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("name", String(30)),
    Column("fullname", String),
)

In [19]:
user_table.c.name

Column('name', String(length=30), table=<user_account>)

In [20]:
user_table.c.keys()

['id', 'name', 'fullname']

In [21]:
user_table.primary_key

PrimaryKeyConstraint(Column('id', Integer(), table=<user_account>, primary_key=True, nullable=False))

In [22]:
# create a second table
from sqlalchemy import ForeignKey

address_table = Table(
    "address",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("user_id", ForeignKey("user_account.id"), nullable=False),
    Column("email_address", String, nullable=False)
)

In [23]:
metadata_obj.create_all(engine)

2026-01-25 12:35:09,051 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:09,053 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2026-01-25 12:35:09,054 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-25 12:35:09,056 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("user_account")
2026-01-25 12:35:09,056 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-25 12:35:09,058 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2026-01-25 12:35:09,060 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-25 12:35:09,063 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("address")
2026-01-25 12:35:09,064 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-25 12:35:09,066 INFO sqlalchemy.engine.Engine 
CREATE TABLE user_account (
	id INTEGER NOT NULL, 
	name VARCHAR(30), 
	fullname VARCHAR, 
	PRIMARY KEY (id)
)


2026-01-25 12:35:09,067 INFO sqlalchemy.engine.Engine [no key 0.00100s] ()
2026-01-25 12:35:09,070 INFO sqlalchemy.engine.Engine 
C

<h2>Establishing a Declarative Base<h2/>

In [24]:
# create a new class that subclasses the SQLAlchemy DeclarativeBase class

from sqlalchemy.orm import DeclarativeBase

class Base(DeclarativeBase):
    pass

In [25]:
Base.metadata

MetaData()

In [26]:
Base.registry

In [27]:
from typing import List
from typing import Optional
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

class User(Base):
    __tablename__ = "user_account"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[Optional[str]]

    addresses: Mapped[List["Address"]] = relationship(back_populates="user")

    def __repr__(self) -> str:
        return f"User(id={self.id!r}, fullname={self.fullname!r})"
    

class Address(Base):
    __tablename__ = "address"

    id: Mapped[int] = mapped_column(primary_key=True)
    email_address: Mapped[str]
    user_id = mapped_column(ForeignKey("user_account.id"))

    user: Mapped[User] = relationship(back_populates="addresses")

    def __repr__(self) -> str:
        return f"Address(id={self.id!r}, email_address={self.email_address!r})"
        

<h2>Emitting DDL to the database from an ORM mapping<h2/>

In [28]:
Base.metadata.create_all(engine)

2026-01-25 12:35:09,195 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:09,198 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2026-01-25 12:35:09,200 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-25 12:35:09,202 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2026-01-25 12:35:09,204 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-25 12:35:09,206 INFO sqlalchemy.engine.Engine COMMIT


In [29]:
# import requests

# URL = "https://my.api.mockaroo.com/transaction.json?key=13f89f70"

# get_data = requests.get(URL)
# data = get_data.json()
# print(data)

In [30]:
some_table = Table("some_table", metadata_obj, autoload_with=engine)

2026-01-25 12:35:09,244 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 12:35:09,246 INFO sqlalchemy.engine.Engine PRAGMA main.table_xinfo("some_table")
2026-01-25 12:35:09,248 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-25 12:35:09,251 INFO sqlalchemy.engine.Engine SELECT sql FROM  (SELECT * FROM sqlite_master UNION ALL   SELECT * FROM sqlite_temp_master) WHERE name = ? AND type in ('table', 'view')
2026-01-25 12:35:09,253 INFO sqlalchemy.engine.Engine [raw sql] ('some_table',)
2026-01-25 12:35:09,256 INFO sqlalchemy.engine.Engine PRAGMA main.foreign_key_list("some_table")
2026-01-25 12:35:09,259 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-25 12:35:09,261 INFO sqlalchemy.engine.Engine PRAGMA temp.foreign_key_list("some_table")
2026-01-25 12:35:09,263 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-25 12:35:09,265 INFO sqlalchemy.engine.Engine SELECT sql FROM  (SELECT * FROM sqlite_master UNION ALL   SELECT * FROM sqlite_temp_master) WHERE name = ? AND type i

In [31]:
some_table

Table('some_table', MetaData(), Column('x', INTEGER(), table=<some_table>), Column('y', INTEGER(), table=<some_table>), schema=None)

In [32]:
print(some_table)

some_table
